# Cue-level analysis: what is the detector actually looking at?

The two analyses promised in the proposal, driven from `src/analyze.py`. Together they
test the one claim in the report that currently rests on behavior alone: that the `pad`
model reads the black border it created rather than any generative fingerprint.

**Radially averaged spectra** (`analyze spectra`) needs images only - no checkpoints, no
torch, no GPU - so it covers **all four arms**, including the two this account never
trained. The arm identity lives in the cache, not in a model.

**Input-gradient attribution** (`analyze attribution`) needs one `model.pt` per source, so
it covers only **this account's two arms**. That is enough for the claim: border mass is
read against an absolute null (`UNIFORM_BORDER_MASS = 0.4375`), not against another arm,
and the sharpest contrast is *within* `pad` - real photographs get black bars, all seven
generators emit square images and get none. If the model reads the border, `pad`'s border
mass on the real class exceeds its border mass on every generator.

Run the spectra cell first: it is the one that cannot fail for lack of anything.

## Setup

Set `OWNER` in the next cell and nothing else. Same paths as `01_run_matrix.ipynb`: the
cache is read from the shared root, the checkpoints from **this account's own** Drive
backup.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob
from pathlib import Path

# Spectra runs fine on CPU. Attribution wants a GPU: 1,600 forward+backward passes per
# checkpoint is seconds on a T4 and unmeasured on CPU, so `--device cuda` is the default below.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> spectra will still run; for attribution use Runtime > Change runtime type > T4 GPU (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

# Who is running this notebook. Only this account's 28 checkpoints have weights here.
OWNER = "noa"

# Read root, shared between both accounts - identical bytes, nothing copied.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Write root, this account's OWN My Drive.
BACKUP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}"

FIGURES = "results/figures"

assert os.path.isdir(DRIVE), (
    f"not found: {DRIVE}\nCheck it is mounted on THIS account: `ls /content/drive/MyDrive`."
)

print(f"owner   {OWNER}")
print("cache  <-", CACHE, "" if os.path.isdir(CACHE) else "  <- MISSING")
print("ckpts  <-", f"{BACKUP}/runs")

## Get the code

In [ ]:
# Idempotent: clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

## Gate: which arms can each analysis actually cover?

Two independent questions, and they have different answers.

**Spectra** needs pixel shards per strategy. The cache was built once in one account and is
reached from the other through a shortcut, so the thing worth checking is whether all four
strategy directories are visible from *this* account or only the two it trained.

**Attribution** needs `model.pt`, and 14 at seed 0 is a full half of the matrix.

In [ ]:
STRATEGIES = ["center_crop", "random_crop", "rescale", "pad"]

print("cache shards per strategy (spectra covers whatever is listed):")
spectra_arms = []
for s in STRATEGIES:
    n = len(glob.glob(f"{CACHE}/{s}/*.npy"))
    print(f"  {s:<13}{n:>3} shards" + ("" if n else "   <- not visible from this account"))
    if n:
        spectra_arms.append(s)

meta = len(glob.glob(f"{CACHE}/meta/*.npz"))
print(f"  meta         {meta:>3} shards (18 expected: 14 train + 4 validation)")

ckpts = sorted(glob.glob(f"{BACKUP}/runs/*/*/seed0/model.pt"))
attribution_arms = sorted({Path(p).parts[-4] for p in ckpts})
print(f"\n{len(ckpts)} seed-0 checkpoints under {BACKUP}/runs (14 expected for one account)")
for s in attribution_arms:
    print(f"  {s:<13}{sum(1 for p in ckpts if Path(p).parts[-4] == s):>3}")

if not spectra_arms:
    print("\nNO CACHE VISIBLE. Check the Drive shortcut points at", CACHE)
if not ckpts:
    print("\nNO CHECKPOINTS. Is OWNER right, and is Drive mounted on THIS account?")

## 1. Radially averaged spectra

Per (class, generator, strategy): grayscale, subtract the mean, `fft2`, `fftshift`,
magnitude, then average over 64 radial rings and plot log-magnitude against frequency. The
informative curve is **real minus fake**.

**No window function**, deliberately. The `pad` border is a real feature of that arm and a
window would taper it away - which is exactly the thing under test. Say so in the caption if
this figure goes in the report.

The FFT itself is seconds; the cost is the Drive read of one arm at a time.

In [ ]:
spectra_cmd = (
    f'python -m src.analyze spectra --cache-dir "{CACHE}"'
    f' --strategies {" ".join(spectra_arms)}'
    f' --out-dir "{FIGURES}" --n-images 500'
)
print(spectra_cmd, "\n")
!{spectra_cmd}

## 2. Input-gradient attribution

`g = df(x)/dx`, `saliency = max_channel |g|`, averaged over 200 images per generator per
strategy. This is the method taught in L10, slides 53-57 - **not Grad-CAM**, which was never
taught in this course.

The number to read is **border mass**: the fraction of total saliency falling in the outer
16-pixel ring, against the uniform null of 0.4375. Two comparisons matter, in this order:

1. **Within `pad`, real versus each generator.** Only real photographs receive a border. If
   `pad`'s border mass on the real class exceeds every generator, the border-reading account
   in the report is confirmed from the model's internals rather than inferred from behavior.
2. **`pad` versus `random_crop`**, a non-padding control from the same 128 square pipeline.

Passing `--strategies` is not tidiness: `command_attribution` loads the test pool for a
strategy *before* checking whether its checkpoints exist, so without it this loads ~200 MB
for arms it will then skip.

In [ ]:
# Attribution is the only step that wants a GPU. Pick the device from what is actually
# available rather than assuming: loading a checkpoint with --device cuda on a CPU runtime
# fails inside torch.load, which is a confusing place to discover the runtime type.
import torch

if torch.cuda.is_available():
    DEVICE, N_IMAGES = "cuda", 200
else:
    DEVICE, N_IMAGES = "cpu", 100
    print("CPU runtime: falling back to --device cpu and halving --n-images to 100.")
    print("For the full 200, use Runtime > Change runtime type > T4 GPU, then re-run")
    print("the setup and clone cells (the restart unmounts Drive).")

attribution_cmd = (
    f'python -m src.analyze attribution --cache-dir "{CACHE}"'
    f' --results-dir "{BACKUP}/runs"'
    f' --strategies {" ".join(attribution_arms)}'
    f' --seed 0 --n-images {N_IMAGES} --device {DEVICE} --out-dir "{FIGURES}"'
)
print(attribution_cmd)
print()
!{attribution_cmd}

## Read the border mass

In [ ]:
UNIFORM = 0.4375  # a saliency map with no structure puts this much mass in the outer ring

path = Path(FIGURES) / "attribution.json"
if not path.exists():
    print("attribution.json not found - run the cell above first.")
else:
    data = json.loads(path.read_text())
    print(f"border mass, against the uniform null {UNIFORM}\n")
    print(f"{'strategy':<13}{'tag':<13}{'border mass':>12}{'vs null':>10}")
    for strategy, block in sorted(data.items()):
        if not isinstance(block, dict):
            continue
        for tag, value in sorted(block.items()):
            mass = value.get("border_mass") if isinstance(value, dict) else value
            if isinstance(mass, (int, float)):
                print(f"{strategy:<13}{tag:<13}{mass:>12.4f}{mass - UNIFORM:>+10.4f}")
    print("\nThe claim is confirmed if, under pad, REAL sits clearly above every generator.")

## Hand off

The JSON and the figures go through git, like `metrics.json`. They are small and they are
what let the other account, and a grader, see the result without the cache or the weights.

Once these exist, §6 of the report should be rewritten: the paragraph headed
**"Cue-level evidence, not obtained"** is a disclosure of a promised analysis that was not
run, and it becomes wrong the moment this notebook succeeds. Replace it with the border-mass
numbers and the spectra reading, and drop the matching sentence from the Limitations
paragraph in §7.

In [ ]:
!git status --short results/figures
print("\nThen, from a terminal with push access:")
print("  git add results/figures/spectra.json results/figures/spectra.png \\")
print("          results/figures/attribution.json results/figures/attribution*.png")
print(f'  git commit -m "Cue-level analysis: spectra (all arms) and attribution ({OWNER} half)"')
print("  git push")